# M4.2 | From distribution shape to feature selection

## Skew, kurtosis, transformation and a safer machine learning pipeline

**Teaching time:** 3 hours  
**Breaks:** Two 10-minute breaks  
**Level:** Beginner  
**Scenario:** Predicting whether a subscription customer will leave next month

This notebook is designed to run from top to bottom.

You will work with synthetic customer data, so there is no confidential or personal information in the dataset.

## Why this matters at work

Raw data rarely arrives in a shape that is ideal for modelling.

Some numerical features may:

- have a long tail
- contain unusually extreme values
- use very different scales
- duplicate information from another feature
- add little predictive value

**Skew and kurtosis** help us describe the shape of a feature.

**Feature-selection methods** help us investigate whether a feature contains useful information about the target.

These are related decisions, but they are not the same decision.

> **Key distinction**
>
> High skew or kurtosis is not a reason to remove a feature automatically.
>
> It is a reason to inspect the feature and consider transformation, outlier handling or a different model.

## Learning outcomes

By the end of this notebook, you will be able to:

1. explain what skew and kurtosis reveal about a numerical feature
2. calculate and interpret skew and excess kurtosis in pandas
3. decide whether a feature needs further investigation or transformation
4. compare a log transformation with a Yeo–Johnson transformation
5. explain why transformations must be fitted on training data only
6. place preprocessing and feature selection inside a scikit-learn pipeline
7. use mutual information to select a smaller feature set
8. compare model performance before and after transformation and feature selection
9. document a defensible feature decision for a stakeholder

## Three-hour learning plan

| Time | Duration | Section |
|---|---:|---|
| 00:00–00:10 | 10 min | Welcome, purpose and learning map |
| 00:10–00:25 | 15 min | Scenario, data and train/test split |
| 00:25–00:45 | 20 min | Describe and visualise distributions |
| 00:45–01:00 | 15 min | Understand skew |
| 01:00–01:15 | 15 min | Understand kurtosis |
| 01:15–01:25 | 10 min | **Break 1** |
| 01:25–01:50 | 25 min | Test transformation choices |
| 01:50–02:05 | 15 min | Prevent leakage with a pipeline |
| 02:05–02:25 | 20 min | Compare baseline and transformed models |
| 02:25–02:35 | 10 min | **Break 2** |
| 02:35–02:55 | 20 min | Select features inside the pipeline |
| 02:55–03:00 | 5 min | Reflection and exit check |

The optional cross-validation appendix is not part of the core three-hour session.

<details>
<summary><strong>Facilitator quick start</strong></summary>

### Before the session

- Run the notebook from top to bottom once.
- Check that pandas, NumPy, matplotlib and scikit-learn are installed.
- Clear the outputs before the live session.
- Ask learners to predict results before running key cells.

### Keep returning to three questions

1. **Shape:** What does the feature's distribution look like?
2. **Relevance:** Does the feature contain information about the target?
3. **Performance:** Does the complete pipeline work better on unseen data?

### Common misconception

A logistic-regression model does not require every feature to be normally distributed.

Transformation may still help by reducing the influence of extreme values, improving linear relationships or putting values on a more useful scale. It should be tested rather than applied automatically.

</details>

# Part 1 | Set up the analysis

## Step 1: Import the tools

This cell contains all imports used in the notebook.

Run it once before continuing.

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.feature_selection import (
    SelectKBest,
    VarianceThreshold,
    mutual_info_classif,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler

RANDOM_STATE = 42

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

print('Setup complete.')

## Step 2: Create the synthetic customer dataset

A subscription business wants to identify customers who may leave next month.

The target is:

- `1`: the customer leaves
- `0`: the customer stays

The dataset deliberately includes:

- right-skewed features
- a feature with heavy tails
- two strongly correlated features
- an irrelevant random feature
- a constant feature
- a customer identifier that must not enter the model

This gives us realistic preprocessing and feature-selection decisions.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
number_of_customers = 1_200

tenure_months = np.clip(
    rng.exponential(scale=26, size=number_of_customers),
    1,
    120,
)

monthly_spend = (
    rng.gamma(shape=2.0, scale=35, size=number_of_customers) + 5
)

support_tickets = rng.poisson(
    lam=1.6,
    size=number_of_customers,
)

satisfaction_score = np.clip(
    rng.normal(loc=7.2, scale=1.4, size=number_of_customers),
    1,
    10,
)

logins_30d = rng.poisson(
    lam=12,
    size=number_of_customers,
)

payment_delays = rng.negative_binomial(
    n=2,
    p=0.62,
    size=number_of_customers,
)

usage_change_pct = rng.normal(
    loc=0,
    scale=16,
    size=number_of_customers,
)

discount_pct = (
    rng.beta(a=1.5, b=5, size=number_of_customers) * 50
)

account_age_months = np.clip(
    tenure_months + rng.normal(0, 2, number_of_customers),
    1,
    130,
)

heavy_tail_noise = (
    rng.standard_t(df=2.2, size=number_of_customers) * 10
)

random_noise = rng.normal(
    loc=0,
    scale=1,
    size=number_of_customers,
)

legacy_plan_flag = np.ones(
    number_of_customers,
    dtype=int,
)

raw_churn_score = (
    -1.20
    - 0.35 * (satisfaction_score - 7)
    + 0.22 * support_tickets
    + 0.28 * payment_delays
    - 0.018 * usage_change_pct
    - 0.012 * (tenure_months - 24)
    + 0.0025 * (monthly_spend - 70)
    - 0.030 * (logins_30d - 12)
    - 0.010 * (discount_pct - 12)
)

churn_probability = 1 / (1 + np.exp(-raw_churn_score))
churn = rng.binomial(n=1, p=churn_probability)

customer_data = pd.DataFrame(
    {
        'customer_id': [
            f'CUST-{number:05d}'
            for number in range(1, number_of_customers + 1)
        ],
        'tenure_months': tenure_months,
        'monthly_spend': monthly_spend,
        'support_tickets': support_tickets,
        'satisfaction_score': satisfaction_score,
        'logins_30d': logins_30d,
        'payment_delays': payment_delays,
        'usage_change_pct': usage_change_pct,
        'discount_pct': discount_pct,
        'account_age_months': account_age_months,
        'heavy_tail_noise': heavy_tail_noise,
        'random_noise': random_noise,
        'legacy_plan_flag': legacy_plan_flag,
        'churn': churn,
    }
)

print('Dataset created:', customer_data.shape)
display(customer_data.head())

## Data dictionary

| Column | Meaning | Initial modelling note |
|---|---|---|
| `customer_id` | Unique customer reference | Identifier: exclude from model |
| `tenure_months` | Months since joining | Likely right-skewed |
| `monthly_spend` | Average monthly spend | Positive and right-skewed |
| `support_tickets` | Support requests | Count variable |
| `satisfaction_score` | Score from 1 to 10 | Bounded numerical feature |
| `logins_30d` | Logins in the last 30 days | Count variable |
| `payment_delays` | Recent late payments | Count variable and likely skewed |
| `usage_change_pct` | Percentage change in usage | Can be positive or negative |
| `discount_pct` | Discount percentage | Bounded at zero |
| `account_age_months` | Age of account | Closely related to tenure |
| `heavy_tail_noise` | Demonstration noise feature | Contains extreme values |
| `random_noise` | Random normal feature | Should have little relevance |
| `legacy_plan_flag` | Constant value of one | No variation |
| `churn` | Target: leave or stay | Do not include in input features |

## Step 3: Check the data before modelling

Before calculating statistics, confirm:

- the dataset has the expected shape
- column types are sensible
- missing values are understood
- the target contains both classes

This synthetic dataset has no missing values, allowing us to focus on distribution shape and feature selection.

In [ ]:
data_quality = pd.DataFrame(
    {
        'data_type': customer_data.dtypes.astype(str),
        'missing_values': customer_data.isna().sum(),
        'unique_values': customer_data.nunique(),
    }
)

display(data_quality)

target_balance = (
    customer_data['churn']
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: 'stays', 1: 'leaves'})
    .to_frame('proportion')
)

display(target_balance)

## Step 4: Separate the target and split the data

We remove:

- `customer_id`, because an identifier is not a meaningful predictor
- `churn`, because it is the answer the model is learning to predict

We then create:

- `X_train` and `y_train`: used to learn preprocessing and model parameters
- `X_test` and `y_test`: held back for final evaluation

> From this point onward, distribution and transformation decisions use the training data.

In [ ]:
feature_columns = [
    column
    for column in customer_data.columns
    if column not in ['customer_id', 'churn']
]

X = customer_data[feature_columns].copy()
y = customer_data['churn'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Training features:', X_train.shape)
print('Test features:', X_test.shape)
print('Training target:', y_train.shape)
print('Test target:', y_test.shape)

### Check your understanding

Why is `customer_id` excluded?

A strong response should mention that:

- it identifies a record rather than describing customer behaviour
- it may create accidental patterns that do not generalise
- it would not give the business a meaningful intervention

# Part 2 | Describe distribution shape

## Step 5: Start with familiar descriptive statistics

Mean, median and standard deviation describe the centre and spread of a feature.

They do not tell the full story.

Two features can have similar averages while having very different:

- tails
- asymmetry
- extreme values
- concentrations of observations

In [ ]:
descriptive_summary = (
    X_train.describe()
    .T[['mean', '50%', 'std', 'min', 'max']]
    .rename(columns={'50%': 'median'})
)

descriptive_summary['mean_minus_median'] = (
    descriptive_summary['mean']
    - descriptive_summary['median']
)

display(
    descriptive_summary.sort_values(
        'mean_minus_median',
        key=lambda values: values.abs(),
        ascending=False,
    )
)

## Step 6: Look at the distributions

We will compare three features:

- `monthly_spend`: expected to be right-skewed
- `usage_change_pct`: expected to be closer to symmetric
- `heavy_tail_noise`: expected to contain extreme values

For each chart, compare the mean with the median.

In [ ]:
features_to_plot = [
    'monthly_spend',
    'usage_change_pct',
    'heavy_tail_noise',
]

for feature in features_to_plot:
    values = X_train[feature]

    plt.figure(figsize=(8, 3.5))
    plt.hist(values, bins=30)
    plt.axvline(
        values.mean(),
        linestyle='--',
        label='Mean',
    )
    plt.axvline(
        values.median(),
        linestyle=':',
        label='Median',
    )
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Number of customers')
    plt.legend()
    plt.tight_layout()
    plt.show()

### Pause and predict

Before calculating skew and kurtosis, discuss:

1. Which feature looks most asymmetric?
2. Which feature appears to have the most extreme observations?
3. Would you remove either feature based only on these charts?
4. What additional evidence would you want?

# Part 3 | Understand skew

## What is skew?

Skew describes the **asymmetry** of a distribution.

- **Positive skew:** a longer tail extends towards larger values.
- **Negative skew:** a longer tail extends towards smaller values.
- **Skew near zero:** the distribution is more balanced around its centre.

Practical interpretation guide:

| Absolute skew | Working description |
|---:|---|
| Below 0.5 | Roughly symmetric |
| 0.5 to below 1.0 | Moderately skewed |
| 1.0 or above | Strongly skewed |

These thresholds are prompts for investigation, not universal pass/fail rules.

In [ ]:
def skew_band(skew_value: float) -> str:
    absolute_skew = abs(skew_value)

    if absolute_skew < 0.5:
        return 'roughly symmetric'

    if absolute_skew < 1.0:
        return 'moderately skewed'

    return 'strongly skewed'


skew_table = (
    X_train.skew()
    .rename('skew')
    .to_frame()
)

skew_table['absolute_skew'] = skew_table['skew'].abs()
skew_table['interpretation'] = (
    skew_table['skew'].apply(skew_band)
)

skew_table = skew_table.sort_values(
    'absolute_skew',
    ascending=False,
)

display(skew_table)

## What skew can and cannot tell us

Skew can help us decide whether to:

- inspect a feature more closely
- test a power transformation
- compare mean and median
- investigate unusual values
- choose a model less sensitive to distribution shape

Skew does **not** tell us:

- whether the feature predicts the target
- whether the feature should be removed
- whether the data is wrong
- whether transformation will improve the model

### Learner activity: make an initial transformation shortlist

Using `skew_table`, identify:

1. one strongly skewed feature
2. one roughly symmetric feature
3. one count feature where zeros have a real business meaning

For each feature, choose one initial action:

- leave unchanged for now
- inspect values
- test a transformation
- remove because it has no variation

Be ready to explain your evidence.

# Part 4 | Understand kurtosis

## What is kurtosis?

Kurtosis describes the weight of the tails of a distribution and its tendency to contain extreme values.

In pandas, `.kurt()` reports **excess kurtosis**:

- a normal distribution is approximately `0`
- a positive value suggests heavier tails
- a negative value suggests lighter tails or a flatter distribution

Practical interpretation guide:

| Excess kurtosis | Working description |
|---:|---|
| Below -1 | Lighter tails or flatter shape |
| -1 to 1 | Closer to normal-like tail weight |
| Above 1 | Heavier tails |
| Well above 3 | Inspect extreme values carefully |

Kurtosis can be unstable in small samples and can be strongly affected by a few observations.

In [ ]:
def kurtosis_band(kurtosis_value: float) -> str:
    if kurtosis_value < -1:
        return 'lighter tails or flatter shape'

    if kurtosis_value <= 1:
        return 'near normal-like tail weight'

    if kurtosis_value <= 3:
        return 'heavier tails'

    return 'very heavy tails: inspect extremes'


shape_table = pd.DataFrame(
    {
        'skew': X_train.skew(),
        'excess_kurtosis': X_train.kurt(),
    }
)

shape_table['skew_interpretation'] = (
    shape_table['skew'].apply(skew_band)
)

shape_table['kurtosis_interpretation'] = (
    shape_table['excess_kurtosis'].apply(kurtosis_band)
)

shape_table['shape_priority'] = np.maximum(
    shape_table['skew'].abs(),
    shape_table['excess_kurtosis'].abs(),
)

shape_table = shape_table.sort_values(
    'shape_priority',
    ascending=False,
)

display(shape_table)

## Do not confuse shape with relevance

Consider these two demonstration features:

- `heavy_tail_noise` may have high kurtosis but no useful relationship with churn.
- `monthly_spend` may be skewed and still contain useful predictive information.

This is why a feature decision needs more than one statistic.

| Question | Evidence |
|---|---|
| Is the feature usable? | Data type, missingness and variation |
| Does its shape need attention? | Plot, skew and kurtosis |
| Is it related to another feature? | Correlation and domain knowledge |
| Is it related to the target? | Feature-selection score |
| Does it help the pipeline? | Validation or test performance |

In [ ]:
worked_decision_log = pd.DataFrame(
    {
        'feature': [
            'monthly_spend',
            'usage_change_pct',
            'heavy_tail_noise',
            'legacy_plan_flag',
        ],
        'shape_evidence': [
            'Review skew table',
            'Review skew table',
            'Review kurtosis table',
            'No variation',
        ],
        'initial_action': [
            'Test a power transformation',
            'Leave unchanged initially',
            'Inspect extremes; do not drop automatically',
            'Remove before modelling',
        ],
    }
)

display(worked_decision_log)

# Break 1 | 10 minutes

You have completed:

- train/test separation
- descriptive statistics
- visual inspection
- skew
- kurtosis
- an initial decision log

When you return, we will test transformations rather than assuming they help.

# Part 5 | Test transformation choices

## Why transform a feature?

A transformation changes the numerical representation of a feature.

It may help by:

- compressing a long right tail
- reducing the leverage of large values
- making a relationship more linear
- placing values on a more useful scale

It does not automatically:

- remove outliers
- create predictive information
- correct inaccurate data
- improve every model

Tree-based models are often less sensitive to monotonic transformations than linear or distance-based models.

## Transformation decision guide

| Method | Suitable for | Important caution |
|---|---|---|
| `log1p` | Non-negative, right-skewed values including zero | Cannot be applied directly to negative values |
| Box–Cox | Strictly positive numerical values | Does not accept zero or negative values |
| Yeo–Johnson | Positive, zero and negative values | Must learn its parameters from training data |
| StandardScaler | Changing centre and scale | Does not remove skew |
| Clipping or winsorising | Documented extreme-value treatment | Can hide genuine rare events |

We will compare `log1p` and Yeo–Johnson.

## Step 7: Test `log1p` on monthly spend

`np.log1p(x)` calculates `log(1 + x)`.

Adding one allows the transformation to handle zero.

This exploratory comparison uses the training data only.

In [ ]:
monthly_spend_original = X_train['monthly_spend']
monthly_spend_log1p = np.log1p(monthly_spend_original)

log_comparison = pd.DataFrame(
    {
        'version': ['original', 'log1p'],
        'skew': [
            monthly_spend_original.skew(),
            monthly_spend_log1p.skew(),
        ],
        'excess_kurtosis': [
            monthly_spend_original.kurt(),
            monthly_spend_log1p.kurt(),
        ],
    }
).set_index('version')

display(log_comparison)

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(monthly_spend_original, bins=30)
plt.title('Monthly spend before transformation')
plt.xlabel('Monthly spend')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(monthly_spend_log1p, bins=30)
plt.title('Monthly spend after log1p transformation')
plt.xlabel('log(1 + monthly spend)')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.show()

### Interpret the result

Discuss:

1. Did the transformation reduce absolute skew?
2. Did it reduce excess kurtosis?
3. Has the business meaning become less direct?
4. Would you transform the feature for every model?
5. What model evidence would you collect before deciding?

## Step 8: Preview a Yeo–Johnson transformation

Yeo–Johnson is useful when a feature may contain:

- positive values
- zeros
- negative values

Unlike a fixed log transformation, it estimates a parameter for each feature.

We use `standardize=False` in this preview so we can focus on the change in shape.

In [ ]:
preview_features = [
    'monthly_spend',
    'payment_delays',
    'usage_change_pct',
    'heavy_tail_noise',
]

power_preview = PowerTransformer(
    method='yeo-johnson',
    standardize=False,
)

preview_array = power_preview.fit_transform(
    X_train[preview_features]
)

preview_transformed = pd.DataFrame(
    preview_array,
    columns=preview_features,
    index=X_train.index,
)

power_shape_comparison = pd.DataFrame(
    {
        'before_skew': X_train[preview_features].skew(),
        'after_skew': preview_transformed.skew(),
        'before_excess_kurtosis': X_train[preview_features].kurt(),
        'after_excess_kurtosis': preview_transformed.kurt(),
        'learned_lambda': power_preview.lambdas_,
    }
)

power_shape_comparison['absolute_skew_reduction'] = (
    power_shape_comparison['before_skew'].abs()
    - power_shape_comparison['after_skew'].abs()
)

display(power_shape_comparison)

## What to notice

Yeo–Johnson may improve some features more than others.

A feature with very heavy tails may still have high kurtosis after transformation.

That is useful evidence:

- the method is not a universal cure
- unusual values may need separate investigation
- a robust or tree-based model may be worth comparing
- feature relevance must still be assessed separately

The preview transformer above will not be used for final modelling. We will create a fresh transformer inside a pipeline.

# Part 6 | Prevent leakage with a pipeline

## What is data leakage?

Data leakage occurs when information from outside the training data influences model fitting.

A transformer learns information such as:

- a mean
- a standard deviation
- a Yeo–Johnson lambda
- a feature-selection score

If we fit those steps before the train/test split, the test data has influenced the model-building process.

A pipeline keeps the sequence together:

1. remove unusable features
2. transform the remaining features
3. select features when required
4. fit the model
5. apply the learned steps consistently to new data

## Step 9: Create a reusable evaluation function

We will compare:

- accuracy
- recall for customers who leave
- ROC AUC

ROC AUC measures how well the model ranks customers across possible classification thresholds.

Recall answers:

> Of the customers who actually leave, how many did the model identify?

In [ ]:
def evaluate_pipeline(
    pipeline_name: str,
    pipeline: Pipeline,
) -> dict:
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    return {
        'pipeline': pipeline_name,
        'accuracy': accuracy_score(y_test, predictions),
        'recall_churn': recall_score(y_test, predictions),
        'roc_auc': roc_auc_score(y_test, probabilities),
    }


print('Evaluation function ready.')

## Step 10: Build two pipelines

### Baseline pipeline

1. Remove features with no variation.
2. Standardise the numerical features.
3. Fit logistic regression.

### Power-transformed pipeline

1. Remove features with no variation.
2. Apply Yeo–Johnson and standardisation.
3. Fit logistic regression.

`StandardScaler` changes centre and scale. It does not correct skew.

In [ ]:
baseline_pipeline = Pipeline(
    steps=[
        (
            'variance',
            VarianceThreshold(threshold=0.0),
        ),
        (
            'scale',
            StandardScaler(),
        ),
        (
            'model',
            LogisticRegression(
                max_iter=2_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

power_pipeline = Pipeline(
    steps=[
        (
            'variance',
            VarianceThreshold(threshold=0.0),
        ),
        (
            'power',
            PowerTransformer(
                method='yeo-johnson',
                standardize=True,
            ),
        ),
        (
            'model',
            LogisticRegression(
                max_iter=2_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print('Pipelines created.')

## Step 11: Compare baseline and transformation

Do not assume the transformed pipeline must win.

A useful experiment may show:

- better performance
- similar performance
- worse performance

All three outcomes provide evidence for a decision.

In [ ]:
initial_results = [
    evaluate_pipeline(
        'Baseline: scale only',
        baseline_pipeline,
    ),
    evaluate_pipeline(
        'Yeo-Johnson transformation',
        power_pipeline,
    ),
]

initial_results_table = (
    pd.DataFrame(initial_results)
    .set_index('pipeline')
)

display(initial_results_table.round(3))

### Debrief

Use the results to answer:

1. Which pipeline has the higher ROC AUC?
2. Which pipeline identifies more customers who leave?
3. Is the difference large enough to matter?
4. Does the transformed pipeline make the workflow less interpretable?
5. What other evidence would you collect before production use?

A transformed feature is not automatically a better feature.

The decision should balance:

- predictive performance
- stability
- interpretability
- maintenance
- business impact

# Break 2 | 10 minutes

You have completed:

- a fixed log transformation
- a Yeo–Johnson preview
- leakage-safe pipelines
- a baseline comparison

When you return, you will add feature selection to the pipeline.

# Part 7 | Select features inside the pipeline

## Feature selection asks a different question

Skew and kurtosis describe the feature itself.

Feature selection investigates whether the feature helps explain or predict the target.

We will use three layers:

1. **Variance:** Does the feature change at all?
2. **Correlation review:** Does it strongly duplicate another feature?
3. **Mutual information:** Does it contain information about churn?

Mutual information can identify linear and non-linear relationships.

A score near zero suggests little detectable relationship, but it is not proof that the feature is useless in every model.

## Step 12: Review variance and correlation

The constant `legacy_plan_flag` should have zero variance.

`tenure_months` and `account_age_months` should be strongly correlated.

Correlation is not an automatic removal rule. Domain knowledge helps decide which feature is clearer, more reliable or available at prediction time.

In [ ]:
variance_review = (
    X_train.var()
    .sort_values()
    .rename('variance')
    .to_frame()
)

display(variance_review)

In [ ]:
correlation_input = X_train.drop(
    columns=['legacy_plan_flag']
)

correlation_matrix = correlation_input.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1,
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
    .rename('absolute_correlation')
    .reset_index()
    .rename(
        columns={
            'level_0': 'feature_a',
            'level_1': 'feature_b',
        }
    )
)

display(correlation_pairs.head(8))

## Step 13: Build a transformation and selection pipeline

The pipeline will:

1. remove constant features
2. fit Yeo–Johnson on the training data
3. calculate mutual-information scores
4. retain the six highest-scoring features
5. fit logistic regression

Feature selection belongs inside the pipeline because its scores are learned from data.

In [ ]:
mutual_information_score = partial(
    mutual_info_classif,
    random_state=RANDOM_STATE,
)

selected_pipeline = Pipeline(
    steps=[
        (
            'variance',
            VarianceThreshold(threshold=0.0),
        ),
        (
            'power',
            PowerTransformer(
                method='yeo-johnson',
                standardize=True,
            ),
        ),
        (
            'select',
            SelectKBest(
                score_func=mutual_information_score,
                k=6,
            ),
        ),
        (
            'model',
            LogisticRegression(
                max_iter=2_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

selection_result = evaluate_pipeline(
    'Yeo-Johnson + six selected features',
    selected_pipeline,
)

comparison_results = (
    pd.DataFrame(initial_results + [selection_result])
    .set_index('pipeline')
)

display(comparison_results.round(3))

## Step 14: Inspect what the selector retained

A pipeline reduces leakage, but it should not hide the reasoning.

We will retrieve:

- the features left after variance filtering
- each feature's mutual-information score
- whether the selector retained it
- its original training-data skew
- its original training-data excess kurtosis

This table connects distribution shape with target relevance without treating them as the same measure.

In [ ]:
variance_mask = (
    selected_pipeline
    .named_steps['variance']
    .get_support()
)

remaining_features = np.array(X_train.columns)[variance_mask]

selector = selected_pipeline.named_steps['select']
selected_mask = selector.get_support()
selected_features = remaining_features[selected_mask]

feature_audit = pd.DataFrame(
    {
        'feature': remaining_features,
        'mutual_information': selector.scores_,
        'selected': selected_mask,
        'training_skew': (
            shape_table
            .loc[remaining_features, 'skew']
            .to_numpy()
        ),
        'training_excess_kurtosis': (
            shape_table
            .loc[remaining_features, 'excess_kurtosis']
            .to_numpy()
        ),
    }
)

feature_audit = feature_audit.sort_values(
    ['selected', 'mutual_information'],
    ascending=[False, False],
)

print('Selected features:')
for feature in selected_features:
    print('-', feature)

display(feature_audit)

## Interpret the feature audit

Look for examples of the following:

- a skewed feature that was selected
- a skewed feature that was not selected
- a high-kurtosis feature with a low relevance score
- two correlated features that were both retained
- a roughly symmetric feature with predictive value

> Distribution statistics inform preprocessing.  
> Feature-selection scores inform relevance.  
> Test performance informs the final pipeline decision.

## Final learner activity: make a defensible recommendation

Complete the decision template.

Choose:

1. one feature to transform
2. one feature to retain unchanged initially
3. one feature to remove or review

Use evidence from:

- skew
- excess kurtosis
- variance
- correlation
- mutual information
- model performance

Your explanation should be understandable to a non-technical stakeholder.

In [ ]:
decision_template = pd.DataFrame(
    {
        'decision': [
            'Transform',
            'Retain unchanged initially',
            'Remove or review',
        ],
        'feature': ['', '', ''],
        'evidence': ['', '', ''],
        'risk_or_trade_off': ['', '', ''],
        'next_test': ['', '', ''],
    }
)

display(decision_template)

### Success criteria

A strong response:

- names a specific feature
- uses at least two pieces of evidence
- separates shape from predictive relevance
- avoids saying that skew automatically makes data bad
- avoids removing a feature solely because it has extreme values
- proposes a test rather than claiming certainty
- explains the decision in plain English

# Reflection and exit check

Answer these questions without running more code.

1. Does high skew mean a feature should be removed?
2. Why is pandas kurtosis described as excess kurtosis?
3. Which transformation can directly accept negative values?
4. Why does `StandardScaler` not solve skew?
5. Why must feature selection be inside the pipeline?
6. What evidence would justify keeping a feature with high kurtosis?
7. Why might a six-feature model be preferable if its ROC AUC is similar to the full model?

<details>
<summary><strong>Exit-check answers</strong></summary>

1. **No.** High skew is a shape signal. The feature may still be predictive.
2. Pandas subtracts three from Pearson kurtosis, so a normal distribution is approximately zero.
3. **Yeo–Johnson** can accept positive values, zero and negative values.
4. Standardisation changes the mean and scale but preserves the distribution's basic shape.
5. The selector learns from data. Keeping it inside the pipeline prevents test or validation information leaking into selection.
6. Evidence might include strong mutual information, improved validation performance, stable measurement and a credible business explanation.
7. A smaller model may be easier to explain, monitor, maintain and deploy while retaining similar predictive performance.

</details>

# Notebook integrity check

Run this after completing all core cells.

It checks that:

- the dataset has the expected size
- there are no missing values
- the target is binary
- the identifier was excluded
- the selector retained six features
- all evaluation metrics are in valid ranges

In [ ]:
assert customer_data.shape[0] == 1_200
assert customer_data.isna().sum().sum() == 0
assert set(y.unique()) == {0, 1}
assert 'customer_id' not in X_train.columns
assert len(selected_features) == 6

for metric in ['accuracy', 'recall_churn', 'roc_auc']:
    assert comparison_results[metric].between(0, 1).all()

print('All notebook integrity checks passed.')

# Optional appendix | Cross-validation

This section is not part of the core three-hour teaching plan.

A single train/test split can be affected by which records happen to enter the test set.

Cross-validation repeats model evaluation across several training and validation folds.

The pipeline is refitted within every fold, so transformation and feature selection remain leakage-safe.

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

pipelines_for_cross_validation = {
    'Baseline: scale only': baseline_pipeline,
    'Yeo-Johnson transformation': power_pipeline,
    'Yeo-Johnson + six selected features': selected_pipeline,
}

cross_validation_rows = []

for pipeline_name, pipeline in pipelines_for_cross_validation.items():
    fold_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cross_validation,
        scoring='roc_auc',
    )

    cross_validation_rows.append(
        {
            'pipeline': pipeline_name,
            'mean_roc_auc': fold_scores.mean(),
            'standard_deviation': fold_scores.std(),
            'minimum_fold': fold_scores.min(),
            'maximum_fold': fold_scores.max(),
        }
    )

cross_validation_results = (
    pd.DataFrame(cross_validation_rows)
    .set_index('pipeline')
)

display(cross_validation_results.round(3))

# Facilitator-only design audit

This is a design-stage self-audit, not a substitute for learner testing.

| Quality criterion | Score |
|---|---:|
| Clear purpose and observable outcomes | 1.00 / 1.00 |
| Exact three-hour pacing with two breaks | 1.00 / 1.00 |
| Beginner-friendly learning sequence | 1.00 / 1.00 |
| Practical workplace scenario | 1.00 / 1.00 |
| Active learner decisions and reflection | 1.00 / 1.00 |
| Short, readable and executable code steps | 1.00 / 1.00 |
| Correct train/test and leakage treatment | 1.00 / 1.00 |
| Clear integration of skew, kurtosis and feature selection | 1.00 / 1.00 |
| Facilitator guidance and misconception handling | 0.90 / 1.00 |
| Accessibility, portability and production readiness | 0.80 / 1.00 |
| **Total** | **9.70 / 10.00** |

The remaining 0.30 is reserved for:

- live learner feedback
- observed timing in delivery
- accessibility testing in the target notebook environment
- adjustments based on the facilitator's client context

## Final takeaway

A reliable feature decision rarely comes from one statistic.

Use:

- skew and kurtosis to understand shape
- plots to see the underlying values
- variance and correlation to identify structural issues
- feature-selection scores to investigate target relevance
- pipelines to prevent leakage
- test or validation performance to compare complete workflows
- domain knowledge to decide what is credible and useful

The aim is not to force every feature into a normal distribution.

The aim is to create a pipeline that is accurate, explainable, reproducible and safe to apply to new data.